# Demo — AgentCore Runtime Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-runtime-deploy/demo-runtime-deploy.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 2: Core Execution (Runtime and Gateway)

Simulates deploying an agent package to the AgentCore Runtime.
Demonstrates how the Runtime provisions an isolated microVM and attaches
the execution role, preventing cross-tenant data leakage.

Prerequisites: AWS credentials configured.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json
import os
import uuid

try:
    import boto3
    HAS_BOTO3 = True
except ImportError:
    HAS_BOTO3 = False

# --- Configuration ---
AGENT_NAME = os.environ.get("AGENT_NAME", "TravelAgent")
EXECUTION_ROLE_ARN = os.environ.get(
    "EXECUTION_ROLE_ARN",
    "arn:aws:iam::123456789012:role/TravelAgentExecutionRole",
)
IMAGE_URI = "123456789012.dkr.ecr.us-east-1.amazonaws.com/travel-agent:v1"

if HAS_BOTO3:
    control = boto3.client("bedrock-agentcore-control")

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def deploy_runtime():
    """Deploy the agent container to AgentCore Runtime.
    This is the boto3 pattern for create_agent_runtime."""
    print_section("Deploying Agent to AgentCore Runtime")

    params = {
        "name": AGENT_NAME,
        "runtimeConfiguration": {
            "imageUri": IMAGE_URI,
            "computeType": "MICRO_VM",
            "vpcConfiguration": {
                "subnetIds": ["subnet-abc", "subnet-xyz"],
                "securityGroupIds": ["sg-12345"],
            }
        },
        "executionRoleArn": EXECUTION_ROLE_ARN,
        "clientToken": str(uuid.uuid4()),
    }

    print("  API: bedrock-agentcore-control.create_agent_runtime")
    print(f"  Parameters:\n{json.dumps(params, indent=4)}")

    # In production, uncomment to actually deploy:
    # response = control.create_agent_runtime(**params)
    # print(f"  Deployed: {response['agentRuntimeArn']}")

    print("\n  [Simulated] Runtime deployment initiated. Status: PROVISIONING")
    return f"arn:aws:bedrock-agentcore:us-east-1:123456789012:agent-runtime/{AGENT_NAME}"

def explain_microvm_isolation():
    """Explain why microVMs are critical for agent security."""
    print_section("Why MicroVM Isolation?")

    print("  Agents generate and execute code dynamically (e.g., Python REPLs).")
    print("  If agents ran on shared compute (like standard Lambda or ECS):")
    print("  1. A malicious prompt could read data from a previous session.")
    print("  2. 'Jailbroken' code could access the underlying host memory.")
    print()
    print("  AgentCore Runtime uses Firecracker MicroVMs:")
    print("  - Every session gets a dedicated, hardware-isolated boundary.")
    print("  - When the session ends, the microVM is destroyed.")
    print("  - Complete prevention of cross-tenant and cross-session data leakage.")

def main():
    print("AgentCore Runtime Deployment — Instructor Demo\n")
    deploy_runtime()
    explain_microvm_isolation()

    print_section("Key Takeaways")
    print("  1. Runtime handles compute provisioning, you just provide the container.")
    print("  2. MicroVM isolation is required because agents execute untrusted dynamic code.")
    print("  3. VPC configuration allows the agent to access private enterprise networks.")

if __name__ == "__main__":
    main()
